<a href="https://colab.research.google.com/github/sasandi123/ayurvedic-skin-disease-treatment/blob/feature%2Fherb-recommendation/personalized_herb_COMPLETE_FIXED_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Personalized Herb Recommendation System


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ═══════════════════════════════════════════════════════════
# IMPORTS
# ═══════════════════════════════════════════════════════════

import pandas as pd
import numpy as np
import joblib
import json
import pickle
import h5py
import datetime
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import MultiLabelBinarizer, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import accuracy_score, hamming_loss, f1_score, make_scorer
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import cross_val_score


print("All imports successful")

In [ ]:
# ═══════════════════════════════════════════════════════════
# CONFIGURATION
# ═══════════════════════════════════════════════════════════

# Update this path to your CSV file
CSV_PATH = "/content/drive/MyDrive/Colab Notebooks/DSGP/ayurvedic_treatment_database_medical_validation_ready (1).csv"

# Output files
MODEL_FILE = "herb_recommendation_rf_model.pkl"
ENCODERS_FILE = "label_encoders.pkl"
MLB_FILE = "mlb_herbs.pkl"
H5_FILE = "herb_ai_knowledge_base.h5"

print("Configuration set")

In [ ]:
# ═══════════════════════════════════════════════════════════
# LOAD AND PREPARE DATA
# ═══════════════════════════════════════════════════════════

import pandas as pd

# Use the CSV_PATH defined in the config cell
FILE_PATH = CSV_PATH # Changed from .gsheet to .csv

df = pd.read_csv(FILE_PATH) # Changed from pd.read_excel to pd.read_csv

print(df.shape)
print(df.head())
print(f"Original Dataset Shape: {df.shape}")
print(f"Columns: {df.columns.tolist()}")

# -------------------------------------------------
# DATA CLEANING (Fix capitalization and spaces)
# -------------------------------------------------

categorical_columns = [
    'disease',
    'severity',
    'age_group',
    'skin_oiliness',
    'skin_sensitivity'
]

for col in categorical_columns:
    df[col] = df[col].astype(str).str.strip().str.lower()

# Select relevant columns
df_filtered = df[[
    'disease', 'severity', 'age_group', 'skin_oiliness', 'skin_sensitivity',
    'herb_name_english'
]].copy()

print(f"Filtered Dataset Shape: {df_filtered.shape}")

# Group by input features and aggregate herbs
df_grouped = df_filtered.groupby(
    ['disease', 'severity', 'age_group', 'skin_oiliness', 'skin_sensitivity']
)['herb_name_english'].apply(list).reset_index()

print(f"Grouped Dataset Shape: {df_grouped.shape}")

# Check unique herbs
all_herbs = set()
for herbs in df_grouped['herb_name_english']:
    all_herbs.update(herbs)

print(f"Number of Unique Herbs: {len(all_herbs)}")
print(f"Herbs: {sorted(all_herbs)}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# FEATURE ENCODING
# ═══════════════════════════════════════════════════════════

# Create label encoders for categorical features
label_encoders = {}
feature_columns = ['disease', 'severity', 'age_group', 'skin_oiliness', 'skin_sensitivity']

X_encoded = df_grouped[feature_columns].copy()

for col in feature_columns:
    le = LabelEncoder()
    X_encoded[col] = le.fit_transform(df_grouped[col])
    label_encoders[col] = le
    print(f"{col}: {dict(zip(le.classes_, le.transform(le.classes_)))}")

# Multi-label binarizer for herbs (output)
mlb = MultiLabelBinarizer()
y_encoded = mlb.fit_transform(df_grouped['herb_name_english'])

print(f"\nX_encoded shape: {X_encoded.shape}")
print(f"y_encoded shape: {y_encoded.shape}")
print(f"Herb classes: {mlb.classes_}")

In [ ]:
# ═══════════════════════════════════════════════════════════
# TRAIN-TEST SPLIT
# ═══════════════════════════════════════════════════════════

X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y_encoded, test_size=0.2, random_state=42
)

print(f"Training set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")

In [ ]:
# ═══════════════════════════════════════════════════════════
# MODEL COMPARISON (FOR REPORT TABLE)
# ═══════════════════════════════════════════════════════════

print("\n" + "="*50)
print("MODEL COMPARISON")
print("="*50)

models = {
    "Logistic Regression": MultiOutputClassifier(LogisticRegression(max_iter=1000)),
    "Support Vector Classifier": MultiOutputClassifier(SVC(kernel='linear')),
    "Random Forest": MultiOutputClassifier(RandomForestClassifier(n_estimators=200, random_state=42)),
    "K-Nearest Neighbours": MultiOutputClassifier(KNeighborsClassifier(n_neighbors=5))
}

results = []

for name, clf in models.items():

    print(f"\nTraining {name}...")

    # Cross validation using first herb label
    # The error occurs here because y_encoded[:,0] results in a 1D array,
    # but MultiOutputClassifier expects a 2D array for its 'y' input.
    # We fix this by selecting the column in a way that preserves its 2D shape (e.g., y_encoded[:, [0]])
    cv_scores = cross_val_score(
        clf,
        X_encoded,
        y_encoded[:,[0]], # Changed from y_encoded[:,0] to y_encoded[:,[0]] to keep it 2D
        cv=5,
        scoring='accuracy'
    )

    cv_mean = cv_scores.mean()

    # Train model
    clf.fit(X_train, y_train)

    # Predictions
    y_pred_model = clf.predict(X_test)

    # Accuracy
    acc = accuracy_score(y_test, y_pred_model)

    print(f"{name} CV Accuracy: {cv_mean:.4f}")
    print(f"{name} Test Accuracy: {acc:.4f}")

    results.append([name, cv_mean, acc])

    # Confusion Matrix (first herb label)
    cm = confusion_matrix(y_test[:,0], y_pred_model[:,0], labels=[0,1])

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=["No Herb", "Herb"]
    )

    plt.figure()
    disp.plot(values_format='d')
    plt.title(f"{name} Confusion Matrix (First Herb)")
    plt.show()

# Convert results to DataFrame
results_df = pd.DataFrame(
    results,
    columns=["Model", "Cross Validation Mean Accuracy", "Model Accuracy"]
)

print("\nModel Comparison Table:")
print(results_df)


In [ ]:
# ═══════════════════════════════════════════════════════════
# MODEL TRAINING WITH GRID SEARCH
# ═══════════════════════════════════════════════════════════

print("\nRunning GridSearchCV...")

# Define parameter grid
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# Base Random Forest
rf_base = RandomForestClassifier(random_state=42)

# Multi-output wrapper
model = MultiOutputClassifier(rf_base)

# Grid search (on first output for simplicity)
rf_grid = RandomForestClassifier(random_state=42)
grid_search = GridSearchCV(
    rf_grid,
    param_grid,
    cv=5,
    scoring='f1_weighted',
    verbose=1,
    n_jobs=-1
)

# Fit on first output to find best params
grid_search.fit(X_train, y_train[:, 0])

print(f"Best Parameters: {grid_search.best_params_}")

# Train final model with best params
best_params = grid_search.best_params_
rf_optimized = RandomForestClassifier(**best_params, random_state=42)
model = MultiOutputClassifier(rf_optimized)

print("\nTraining final model with best parameters...")
model.fit(X_train, y_train)

print("Model training complete!")

In [ ]:
# ═══════════════════════════════════════════════════════════
# MODEL EVALUATION
# ═══════════════════════════════════════════════════════════

# Predictions
# Get probabilities
y_prob = np.array([est.predict_proba(X_test)[:, 1] for est in model.estimators_]).T

#custom threshold
threshold = 0.3
y_pred = (y_prob >= threshold).astype(int)

# Calculate metrics
subset_accuracy = accuracy_score(y_test, y_pred)
hamming = hamming_loss(y_test, y_pred)
micro_f1 = f1_score(y_test, y_pred, average='micro')
macro_f1 = f1_score(y_test, y_pred, average='macro')

print("\n" + "="*40)
print("MODEL PERFORMANCE")
print("="*40)
print(f"Subset Accuracy: {subset_accuracy:.4f}")
print(f"Hamming Loss: {hamming:.4f}")
print(f"Micro F1: {micro_f1:.4f}")
print(f"Macro F1: {macro_f1:.4f}")
print("="*40)

In [ ]:
# ═══════════════════════════════════════════════════════════
# SAVE MODEL FILES (CRITICAL!)
# ═══════════════════════════════════════════════════════════

print("\nSaving model files...")

# 1. Save Random Forest model
joblib.dump(model, MODEL_FILE)
print(f"Model saved: {MODEL_FILE}")

# 2. Save label encoders
with open(ENCODERS_FILE, 'wb') as f:
    pickle.dump(label_encoders, f)
print(f"Encoders saved: {ENCODERS_FILE}")

# 3. Save MultiLabelBinarizer
with open(MLB_FILE, 'wb') as f:
    pickle.dump(mlb, f)
print(f"MLB saved: {MLB_FILE}")

# 4. Save metadata to H5 file
# Extract feature importance from first estimator
feature_importance = model.estimators_[0].feature_importances_

with h5py.File(H5_FILE, 'w') as h5f:
    # Feature importance
    grp_fi = h5f.create_group('feature_importance')
    grp_fi.create_dataset(
        'feature_names',
        data=np.array(feature_columns, dtype='S')
    )
    grp_fi.create_dataset(
        'importance_scores',
        data=feature_importance
    )

    # Model metrics
    grp_metrics = h5f.create_group('model_metrics')
    grp_metrics.create_dataset('subset_accuracy', data=subset_accuracy)
    grp_metrics.create_dataset('hamming_loss', data=hamming)
    grp_metrics.create_dataset('micro_f1', data=micro_f1)
    grp_metrics.create_dataset('macro_f1', data=macro_f1)

    # Training info
    grp_info = h5f.create_group('training_info')
    grp_info.create_dataset('train_samples', data=X_train.shape[0])
    grp_info.create_dataset('test_samples', data=X_test.shape[0])
    grp_info.create_dataset('num_features', data=X_train.shape[1])
    grp_info.create_dataset('num_herbs', data=y_train.shape[1])
    grp_info.create_dataset(
        'timestamp',
        data=datetime.datetime.now().isoformat()
    )

print(f"Metadata saved: {H5_FILE}")

print("\n" + "="*60)
print("ALL FILES SAVED SUCCESSFULLY!")
print("="*60)
print(f"\nGenerated files:")
print(f"  1. {MODEL_FILE}")
print(f"  2. {ENCODERS_FILE}")
print(f"  3. {MLB_FILE}")
print(f"  4. {H5_FILE}")
print("\n Upload these files to your Flask project directory!")
print("="*60)

In [ ]:
# ═══════════════════════════════════════════════════════════# TEST RECOMMENDATION FUNCTION (FIXED!)# ═══════════════════════════════════════════════════════════def recommend_herbs(disease, severity, age_group, skin_oiliness, skin_sensitivity):    """    Recommend herbs based on patient profile        IMPORTANT: All inputs are automatically converted to lowercase        Parameters:    - disease: 'acne', 'eczema', or 'ringworm'    - severity: 'mild', 'moderate', or 'severe'    - age_group: 'child', 'teen', 'young adult', 'adult', or 'senior'    - skin_oiliness: 'oily', 'dry', 'normal', or 'combination'    - skin_sensitivity: 'low', 'normal', 'high', or 'sensitive'    """    # Convert all inputs to lowercase to match training data    input_data = pd.DataFrame([{        'disease': str(disease).lower().strip(),        'severity': str(severity).lower().strip(),        'age_group': str(age_group).lower().strip(),        'skin_oiliness': str(skin_oiliness).lower().strip(),        'skin_sensitivity': str(skin_sensitivity).lower().strip()    }])    # Apply label encoding    for col in feature_columns:        try:            input_data[col] = label_encoders[col].transform(input_data[col])        except ValueError as e:            print(f"❌ Error encoding {col}: {e}")            print(f"   Input value: '{input_data[col].values[0]}'")            print(f"   Valid values: {list(label_encoders[col].classes_)}")            raise    # Predict    prediction = model.predict(input_data)    # Decode herbs    recommended_herbs = mlb.inverse_transform(prediction)[0]    return list(recommended_herbs)# ═══════════════════════════════════════════════════════════# TEST CASES (✅ FIXED - ALL LOWERCASE!)# ═══════════════════════════════════════════════════════════print("\n" + "="*60)print("TESTING RECOMMENDATION FUNCTION")print("="*60)test_cases = [    {        'disease': 'acne',        'severity': 'moderate',        'age_group': 'young adult',      # ✅ FIXED: lowercase        'skin_oiliness': 'oily',          # ✅ FIXED: lowercase        'skin_sensitivity': 'high'        # ✅ FIXED: lowercase    },    {        'disease': 'eczema',        'severity': 'mild',        'age_group': 'adult',             # ✅ FIXED: lowercase        'skin_oiliness': 'dry',           # ✅ FIXED: lowercase        'skin_sensitivity': 'high'        # ✅ FIXED: lowercase    },    {        'disease': 'ringworm',        'severity': 'severe',        'age_group': 'child',             # ✅ FIXED: lowercase        'skin_oiliness': 'normal',        # ✅ FIXED: lowercase        'skin_sensitivity': 'low'         # ✅ FIXED: lowercase    }]print("\n📋 Valid values for reference:")print("  disease: ['acne', 'eczema', 'ringworm']")print("  severity: ['mild', 'moderate', 'severe']")print("  age_group: ['child', 'teen', 'young adult', 'adult', 'senior']")print("  skin_oiliness: ['oily', 'dry', 'normal', 'combination']")print("  skin_sensitivity: ['low', 'normal', 'high', 'sensitive']")print("  (All values must be lowercase!)")for i, test in enumerate(test_cases, 1):    print(f"\nTest {i}:")    print(f"  Input: {test}")    try:        herbs = recommend_herbs(**test)        print(f"  ✅ Recommended Herbs: {herbs}")    except Exception as e:        print(f"  ❌ Error: {e}")print("\n" + "="*60)

In [ ]:
# ═══════════════════════════════════════════════════════════
# FEATURE IMPORTANCE VISUALIZATION
# ═══════════════════════════════════════════════════════════

# Plot feature importance
plt.figure(figsize=(10, 6))
indices = np.argsort(feature_importance)
plt.barh(range(len(feature_importance)), feature_importance[indices])
plt.yticks(range(len(feature_importance)), np.array(feature_columns)[indices])
plt.xlabel('Importance Score')
plt.title('Feature Importance for Herb Recommendation')
plt.tight_layout()
plt.savefig('feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print(" Feature importance plot saved: feature_importance.png")

In [ ]:
# ═══════════════════════════════════════════════════════════
# VERIFICATION - LOAD SAVED MODELS
# ═══════════════════════════════════════════════════════════

print("\n" + "="*60)
print("VERIFYING SAVED FILES")
print("="*60)

# Test loading each file
try:
    loaded_model = joblib.load(MODEL_FILE)
    print(f" {MODEL_FILE} loaded successfully")
except Exception as e:
    print(f" Failed to load {MODEL_FILE}: {e}")

try:
    with open(ENCODERS_FILE, 'rb') as f:
        loaded_encoders = pickle.load(f)
    print(f" {ENCODERS_FILE} loaded successfully")
    print(f"   Encoder keys: {list(loaded_encoders.keys())}")
except Exception as e:
    print(f" Failed to load {ENCODERS_FILE}: {e}")

try:
    with open(MLB_FILE, 'rb') as f:
        loaded_mlb = pickle.load(f)
    print(f" {MLB_FILE} loaded successfully")
    print(f"   Number of herbs: {len(loaded_mlb.classes_)}")
except Exception as e:
    print(f" Failed to load {MLB_FILE}: {e}")

try:
    with h5py.File(H5_FILE, 'r') as h5f:
        print(f"{H5_FILE} loaded successfully")
        print(f"   Groups: {list(h5f.keys())}")
except Exception as e:
    print(f"Failed to load {H5_FILE}: {e}")

print("\n" + "="*60)
print("VERIFICATION COMPLETE!")
print("="*60)
print("\nAll files are ready for Flask integration!")
print("\nNext steps:")
print("  1. Download these 4 files from Colab")
print("  2. Upload to your Flask project directory")
print("  3. Run: python app_with_herb_recommendation.py")
print("  4. Test the integrated system!")
print("\n" + "="*60)


```python
# download all files
from google.colab import files

files.download('herb_recommendation_rf_model.pkl')
files.download('label_encoders.pkl')
files.download('mlb_herbs.pkl')
files.download('herb_ai_knowledge_base.h5')
files.download('feature_importance.png')
```

In [ ]:
from google.colab import files

print("Downloading files...")
files.download(MODEL_FILE)
files.download(ENCODERS_FILE)
files.download(MLB_FILE)
files.download(H5_FILE)
files.download('feature_importance.png')
print("All files downloaded!")